<a href="https://colab.research.google.com/github/PRR-aiexp/CVYoloMLops/blob/main/YoloDagsMlflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install dagshub mlflow ultralytics pandas pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.3/261.3 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/14.5 MB 133.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.9/75

In [3]:
import mlflow
import dagshub
from dagshub import init
from dagshub.auth import add_app_token

In [4]:
DAGSHUB_TOKEN = "2d85e0296e3911b8dd7c7310f5b0968a3b4541b8"  # paste your token

add_app_token(DAGSHUB_TOKEN)

In [5]:
REPO_OWNER = "PRR-aiexp"   # e.g. "PRR-aiexp"
REPO_NAME  = "CVYoloMlops"       # e.g. "CVYoloMlops"

init(
    repo_owner=REPO_OWNER,
    repo_name=REPO_NAME,
    mlflow=True,   # <-- this wires MLflow for you
    dvc=False
)

print("Tracking URI now:", mlflow.get_tracking_uri())

Accessing as PRR-aiexp

Initialized MLflow to track repo "PRR-aiexp/CVYoloMlops"

Repository PRR-aiexp/CVYoloMlops initialized!

Tracking URI now: https://dagshub.com/PRR-aiexp/CVYoloMlops.mlflow


In [18]:
from ultralytics import YOLO
import mlflow
import yaml
import os
from PIL import Image # Import PIL for creating dummy images

# End any active MLflow runs before starting a new one
if mlflow.active_run():
    mlflow.end_run()

# Update the path to the dataset YAML file to reflect the cloned repository structure
DATASET_YAML = "data/yolo_dataset/car_detection.yaml"

# Define the absolute paths for train and validation images and labels
# repo_root is now './' after 'cd CVYoloMlops/'
repo_root = "./"
yolo_dataset_root = os.path.join(repo_root, "data", "yolo_dataset")

# The paths used in the YAML should be relative to the YAML file's location
# or absolute. Given the YAML is in yolo_dataset_root, the paths should be
# relative to yolo_dataset_root.
train_images_path_for_yaml = "images/train"
val_images_path_for_yaml = "images/val"

# For actual directory creation and data generation, use full paths
train_images_full_path = os.path.join(yolo_dataset_root, "images", "train")
val_images_full_path = os.path.join(yolo_dataset_root, "images", "val")
train_labels_full_path = os.path.join(yolo_dataset_root, "labels", "train")
val_labels_full_path = os.path.join(yolo_dataset_root, "labels", "val")


# Function to create dummy image and label files
def create_dummy_data(image_dir, label_dir, num_samples=1):
    os.makedirs(image_dir, exist_ok=True)
    os.makedirs(label_dir, exist_ok=True)
    for i in range(num_samples):
        # Create a dummy image (e.g., a blank white image)
        img = Image.new('RGB', (640, 640), color = 'white')
        img_filename = os.path.join(image_dir, f'dummy_img_{i}.jpg')
        img.save(img_filename)

        # Create a dummy label file
        label_filename = os.path.join(label_dir, f'dummy_img_{i}.txt')
        with open(label_filename, 'w') as f:
            # YOLO format: class_id center_x center_y width height (normalized)
            f.write(f'0 0.5 0.5 0.8 0.8\n') # Class 0 (car), centered, 80% width/height
    print(f"Created {num_samples} dummy image(s) and label(s) in {image_dir} and {label_dir}")

# Create dummy directories and files for training
create_dummy_data(train_images_full_path, train_labels_full_path, num_samples=10) # Create 10 dummy samples for training
create_dummy_data(val_images_full_path, val_labels_full_path, num_samples=2)    # Create 2 dummy samples for validation

# New content for car_detection.yaml with paths relative to the YAML file itself
dataset_config = {
    'train': train_images_path_for_yaml,
    'val': val_images_path_for_yaml,
    'nc': 1, # number of classes, assuming 'car' is the only class
    'names': ['car']
}

# Write the updated dataset configuration to the YAML file in the cloned repo
with open(DATASET_YAML, 'w') as f:
    yaml.dump(dataset_config, f)

# Set MLflow experiment name explicitly
mlflow.set_experiment("YOLOv8 Car Detection Experiment")

with mlflow.start_run(run_name="yolov8n_colab_run1"):

    # Log parameters
    mlflow.log_param("model", "yolov8n")
    mlflow.log_param("imgsz", 640)
    mlflow.log_param("epochs", 5)   # start small
    mlflow.log_param("batch", 8)

    model = YOLO("yolov8n.pt")

    # Define project and name for the training run
    train_project = "runs/train"
    train_name = "car_yolo_dagshub_test"

    results = model.train(
        data=DATASET_YAML,
        imgsz=640,
        epochs=5,        # small test first
        batch=8,
        device=0,        # GPU; use "cpu" if no GPU
        project=train_project,
        name=train_name,
        exist_ok=True
    )

    # Log YOLO metrics
    metrics = results.results_dict
    for k, v in metrics.items():
        try:
            mlflow.log_metric(k, float(v))
        except Exception:
            pass

    # Log best model weights as artifact
    # The best model path is constructed from the project and name of the training run
    best_model_path = os.path.join("/content", "CVYoloMlops", train_project, train_name, "weights", "best.pt")
    mlflow.log_artifact(best_model_path)

print("Done; run should now be in DagsHub MLflow UI.")

🏃 View run nervous-rat-352 at: https://dagshub.com/PRR-aiexp/CVYoloMlops.mlflow/#/experiments/3/runs/a1494a13a0f94d989e06c94ef9e3df36
🧪 View experiment at: https://dagshub.com/PRR-aiexp/CVYoloMlops.mlflow/#/experiments/3
Created 10 dummy image(s) and label(s) in ./data/yolo_dataset/images/train and ./data/yolo_dataset/labels/train
Created 2 dummy image(s) and label(s) in ./data/yolo_dataset/images/val and ./data/yolo_dataset/labels/val
Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/yolo_dataset/car_detection.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, fo

2025/12/09 04:04:27 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2025/12/09 04:04:27 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(9ebd0f66ec5643f89fe381f11bb7c6bb) to https://dagshub.com/PRR-aiexp/CVYoloMlops.mlflow
MLflow: disable with 'yolo settings mlflow=False'
WARNING ⚠️ MLflow: Failed to initialize: INVALID_PARAMETER_VALUE: Response: {'error_code': 'INVALID_PARAMETER_VALUE'}
WARNING ⚠️ MLflow: Not tracking this run
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/CVYoloMlops/runs/train/car_yolo_dagshub_test
Starting training for 5 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        1/5      1.09G      2.292      3.082      2.641          6        640: 100% ━━━━━━━━━━━━ 2/2 5.4it/s 0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 25.1it/s 0.0s
                   all          2          2    0.00333          1     0.0332      0.015

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        2/5       1.1G      2.

In [12]:
#!git clone https://github.com/PRR-aiexp/CVYoloMlops.git
!pwd
%cd CVYoloMlops/

/content
/content/CVYoloMlops
